In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

In [27]:
f = lambda utilization: 226.8324 + 200 * utilization * utilization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import gaussian_kde

# Define slices
slices = ['slice_0', 'slice_1', 'slice_2']
base_path = Path('/home/tngo/MARL4NetworkSlicing/experiments/experiment_5/results/5/test')

# Calculate statistics for each slice
results = {}
for slice_name in slices:
    slice_path = base_path / slice_name
    
    # Read latency, action, and energy data
    latency_df = pd.read_csv(slice_path / 'latency.csv')
    action_df = pd.read_csv(slice_path / 'action.csv')
    energy_df = pd.read_csv(slice_path / 'energy.csv')
    
    # Handle different column names/structures
    latency_vals = latency_df.iloc[:, 0].dropna().values
    energy_vals = energy_df.iloc[:, 0].dropna().values
    
    # Calculate mean latency and average action
    mean_latency = latency_vals.mean() if len(latency_vals) > 0 else np.nan
    avg_action = action_df.values.mean() if action_df.size > 0 else np.nan
    
    results[slice_name] = {
        'mean_latency': mean_latency,
        'avg_action': avg_action,
        'latency_vals': latency_vals,
        'energy_vals': energy_vals
    }

# Display results
print("\n" + "="*60)
print("STATISTICS SUMMARY")
print("="*60)
for slice_name in slices:
    print(f"\n{slice_name}:")
    print(f"  Mean Latency: {results[slice_name]['mean_latency']:.6f}")
    print(f"  Average Action: {results[slice_name]['avg_action']:.6f}")

# Create summary dataframe
summary_df = pd.DataFrame({s: {'Mean Latency': results[s]['mean_latency'], 'Average Action': results[s]['avg_action']} for s in slices}).T
print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)
print(summary_df)

# Plot distributions: first row = latency, second row = energy
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, slice_name in enumerate(slices):
    lat = results[slice_name]['latency_vals']
    en = results[slice_name]['energy_vals']

    ax_lat = axes[0, i]
    if len(lat) > 0:
        ax_lat.hist(lat, bins=30, density=True, alpha=0.6, color=f'C{i}')
        if len(lat) > 1:
            kde_lat = gaussian_kde(lat)
            x_lat = np.linspace(lat.min(), lat.max(), 200)
            ax_lat.plot(x_lat, kde_lat(x_lat), color='k')
    ax_lat.set_title(f'{slice_name} Latency')
    ax_lat.set_xlabel('Latency')

    ax_en = axes[1, i]
    if len(en) > 0:
        ax_en.hist(en, bins=30, density=True, alpha=0.6, color=f'C{i}')
        if len(en) > 1:
            kde_en = gaussian_kde(en)
            x_en = np.linspace(en.min(), en.max(), 200)
            ax_en.plot(x_en, kde_en(x_en), color='k')
    ax_en.set_title(f'{slice_name} Energy')
    ax_en.set_xlabel('Energy')

plt.tight_layout()
plt.show()


STATISTICS SUMMARY

slice_0:
  Mean Latency: 2.270000
  Mean CPU utilization: 0.351949

slice_1:
  Mean Latency: 10.881818
  Mean CPU utilization: 0.202252

slice_2:
  Mean Latency: 1.352352
  Mean CPU utilization: 0.399683

SUMMARY TABLE
         Mean Latency  Average Action
slice_0      2.270000        0.351949
slice_1     10.881818        0.202252
slice_2      1.352352        0.399683


In [29]:
allocation = []
for slice_name in slices:
    slice_path = base_path / slice_name
    
    # Read latency and action data
    allocation_df = pd.read_csv(slice_path / 'allocation.csv')
    allocation.append(allocation_df.loc[:,'mec_0':'mec_4'])


In [34]:
total_alloc = allocation[0]+allocation[1] + allocation[2]

In [36]:
U = total_alloc / 4

In [37]:
E = f(U)

In [39]:
Total_E = np.sum(E, axis = 1)

In [43]:
slice_0_energy = np.sum(E * allocation[0] / total_alloc, axis = 1)
slice_1_energy = np.sum(E * allocation[1] / total_alloc, axis = 1)
slice_2_energy = np.sum(E * allocation[2] / total_alloc, axis = 1)